In [25]:
# Import Spark session helper and PySpark libraries
from spark_session import get_spark_session
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Create Spark session
spark = get_spark_session("sales-etl-pyspark")

# Load CSV and preview data
df = spark.read.csv("data/sales.csv", header=True)
print("Total Row Count:", df.count())
df.show(5)

Total Row Count: 32718
+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+---------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|  CustomerName|        EmailAddress|                Item|Quantity|UnitPrice|TaxAmount|
+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+---------+
|         SO43701|                   1|2019-07-01|   Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99| 271.9992|
|         SO43704|                   1|2019-07-01|    Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99| 269.9992|
|         SO43705|                   1|2019-07-01|     Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99| 271.9992|
|         SO43700|                   1|2019-07-01|  Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982|  55.9279|
|         SO43703|

In [26]:
# Inspect column names and data types
df.dtypes

[('SalesOrderNumber', 'string'),
 ('SalesOrderLineNumber', 'string'),
 ('OrderDate', 'string'),
 ('CustomerName', 'string'),
 ('EmailAddress', 'string'),
 ('Item', 'string'),
 ('Quantity', 'string'),
 ('UnitPrice', 'string'),
 ('TaxAmount', 'string')]

In [27]:
# Count null values in each column
for c in df.columns:
    print(c, df.filter(F.col(c).isNull()).count())

SalesOrderNumber 0
SalesOrderLineNumber 0
OrderDate 0
CustomerName 0
EmailAddress 0
Item 0
Quantity 0
UnitPrice 0
TaxAmount 0


In [28]:
# Cast columns to correct types and rename to snake_case
cast_map = {
    "Quantity": "int",
    "SalesOrderLineNumber": "int",
    "OrderDate": "date",
    "UnitPrice": "float",
    "TaxAmount": "float",
}
for col_name, dtype in cast_map.items():
    df = df.withColumn(col_name, F.col(col_name).cast(dtype))

rename_map = {
    "SalesOrderNumber": "sales_order_number",
    "SalesOrderLineNumber": "sales_order_line_number",
    "OrderDate": "order_date",
    "CustomerName": "customer_name",
    "EmailAddress": "email_address",
    "Item": "item",
    "Quantity": "quantity",
    "UnitPrice": "unit_price",
    "TaxAmount": "tax_amount",
}
for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

df.dtypes

[('sales_order_number', 'string'),
 ('sales_order_line_number', 'int'),
 ('order_date', 'date'),
 ('customer_name', 'string'),
 ('email_address', 'string'),
 ('item', 'string'),
 ('quantity', 'int'),
 ('unit_price', 'float'),
 ('tax_amount', 'float')]

In [29]:
# Top and bottom 5 items by quantity sold
items_by_qty = df.groupBy("item").agg(F.sum("quantity").alias("total_quantity"))

print("Top 5:")
items_by_qty.orderBy(F.desc("total_quantity")).show(5, truncate=False)

print("Bottom 5:")
items_by_qty.orderBy(F.asc("total_quantity")).show(5, truncate=False)

Top 5:
+---------------------+--------------+
|item                 |total_quantity|
+---------------------+--------------+
|Water Bottle - 30 oz.|2097          |
|Patch Kit/8 Patches  |1621          |
|Mountain Tire Tube   |1581          |
|Road Tire Tube       |1212          |
|Sport-100 Helmet, Red|1096          |
+---------------------+--------------+
only showing top 5 rows
Bottom 5:
+-----------------------+--------------+
|item                   |total_quantity|
+-----------------------+--------------+
|Mountain-500 Black, 52 |16            |
|Mountain-500 Silver, 40|16            |
|Mountain-500 Silver, 44|19            |
|Touring-3000 Yellow, 58|21            |
|Touring-3000 Yellow, 62|22            |
+-----------------------+--------------+
only showing top 5 rows


In [30]:
# Yearly revenue, quantity, order count, and average order value
(
    df.groupBy(F.year("order_date").alias("order_year"))
    .agg(
        F.round(F.sum("unit_price"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("sales_order_number").alias("num_orders"),
        F.round(
            F.sum("unit_price") / F.countDistinct("sales_order_number"), 2
        ).alias("avg_order_value"),
    )
    .orderBy("order_year")
    .show()
)

+----------+-------------+--------------+----------+---------------+
|order_year|total_revenue|total_quantity|num_orders|avg_order_value|
+----------+-------------+--------------+----------+---------------+
|      2019|   3863120.23|          1201|      1201|        3216.59|
|      2020|   6372462.18|          2733|      2733|        2331.67|
|      2021|1.069244014E7|         28784|     12525|         853.69|
+----------+-------------+--------------+----------+---------------+



In [31]:
# Monthly revenue and quantity
(
    df.groupBy(F.month("order_date").alias("order_month"))
    .agg(
        F.round(F.sum("unit_price"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
    )
    .orderBy("order_month")
    .show()
)

+-----------+-------------+--------------+
|order_month|total_revenue|total_quantity|
+-----------+-------------+--------------+
|          1|   1144708.02|           491|
|          2|   1024909.02|           423|
|          3|   1229352.46|           526|
|          4|   1177067.86|           524|
|          5|   1269952.75|           578|
|          6|   1348644.15|          1621|
|          7|   2327443.74|          4389|
|          8|   1917298.46|          4379|
|          9|   1972090.11|          4399|
|         10|   2176848.37|          4722|
|         11|   2963850.92|          5649|
|         12|   2375856.69|          5017|
+-----------+-------------+--------------+



In [32]:
# Quarterly revenue
(
    df.groupBy(
        F.year("order_date").alias("order_year"),
        F.quarter("order_date").alias("order_quarter"),
    )
    .agg(F.round(F.sum("unit_price"), 2).alias("total_revenue"))
    .orderBy("order_year", "order_quarter")
    .show()
)

+----------+-------------+-------------+
|order_year|order_quarter|total_revenue|
+----------+-------------+-------------+
|      2019|            3|   1966852.37|
|      2019|            4|   1896267.86|
|      2020|            1|   1895825.09|
|      2020|            2|   1813504.12|
|      2020|            3|   1311858.71|
|      2020|            4|   1351274.27|
|      2021|            1|   1503144.41|
|      2021|            2|   1982160.64|
|      2021|            3|   2938121.24|
|      2021|            4|   4269013.85|
+----------+-------------+-------------+



In [33]:
# Total tax collected by year
(
    df.groupBy(F.year("order_date").alias("order_year"))
    .agg(F.round(F.sum("tax_amount"), 2).alias("total_tax"))
    .orderBy("order_year")
    .show()
)

+----------+---------+
|order_year|total_tax|
+----------+---------+
|      2019|309049.62|
|      2020|509796.99|
|      2021|855395.22|
+----------+---------+

